In [6]:
import scipy
import torch
import numpy as np
torch.manual_seed(42)

In [21]:
X = torch.tensor([[1,2,3],[4,5,6],[7,8,9]])
W = torch.tensor([[1,0],[0,-1]])
X,W

(tensor([[1, 2, 3],
         [4, 5, 6],
         [7, 8, 9]]),
 tensor([[ 1,  0],
         [ 0, -1]]))

In [22]:
scipy.signal.convolve2d(X,W,mode='valid')

array([[4, 4],
       [4, 4]])

### Convolution 2d formula
$S(i, j) = (I * K)(i, j) = \sum_{m} \sum_{n} I(m, n)K(i - m, j - n) $



Horizontal Flip: Reverse the order of the columns.
Vertical Flip: Reverse the order of the rows.

The Result: This is equivalent to rotating the kernel 180°. If your kernel is $K$, the element at top-left $K(0,0)$ moves to the bottom-right.

In [99]:

# m->rows ; n->cols
def convolution2d(I,K):
    """Padding valid and stride 1"""
    k_flipped = torch.flip(K,(0,1))

    i_rows,i_cols = I.shape
    k_rows,k_cols = K.shape

    op_rows = i_rows-k_rows+1
    op_cols = i_cols-k_cols+1

    res = torch.zeros((op_rows, op_cols))

    for row in range(op_rows):
        for col in range(op_cols):
            local_receptive_field = I[row:row+k_rows,col:col+k_cols]
            sub_op = local_receptive_field * k_flipped
            res[row,col]+=torch.sum(sub_op)


    return res
        

In [100]:
X = torch.tensor([[1,2,3],[4,5,6],[7,8,9]])
W = torch.tensor([[1,0],[0,-1]])

convolution2d(X,W)

tensor([[4., 4.],
        [4., 4.]])

Succesfully coded conv2d *(assuming stride 1 and valid padding)

### Cross convolution


$$
S(i, j) = (I * K)(i, j) = \sum_{m} \sum_{n} I(i + m, j + n)K(m, n).
$$


*It is same as convolution but without kernel flipping*

In [107]:
X = torch.tensor([[1,2,3],[4,5,6],[7,8,9]])
W = torch.tensor([[1,0],[0,-1]])

torch.nn.functional.conv2d(X.unsqueeze(0).unsqueeze(0),W.unsqueeze(0).unsqueeze(0))

tensor([[[[-4, -4],
          [-4, -4]]]])

In [112]:

def cross_conv2d(I,K):
    """Assumming stride=1 and valid padding"""
    i_rows,i_cols = I.shape
    k_rows,k_cols = K.shape

    op_rows = i_rows-k_rows+1
    op_cols = i_cols-k_cols+1

    res = torch.zeros((op_rows, op_cols))

    for row in range(op_rows):
        for col in range(op_cols):
            patch = I[row:row+k_rows , col:col+k_cols]
            sub_op = patch * K
            res[row,col]+=torch.sum(sub_op)
    
    return res

    


In [113]:
cross_conv2d(X,W)

tensor([[-4., -4.],
        [-4., -4.]])